# 3. Incident Lifecycle: Assign, Classify, Tune

An investigation isn't finished when you **know** what happened — it's finished when the incident is properly **closed**. SC-200 tests this heavily: you must know the classification taxonomy and how closing an incident feeds learning back into the system.

This notebook walks the lifecycle `New → Active → Closed` with assignment, comments, classification, and rule tuning — plus the bad-vs-best contrast.

> **SC-200 mapping**: "Manage incidents", "Classify and tune analytics rules".


## Setup

This lab reuses the mini-SIEM from Lab 1. Make sure it's running:

```bash
cd ../../01-build-a-siem && docker compose up -d
```

Then, in VS Code:
1. Pick the **`.venv` kernel** from this folder (top-right kernel picker).
2. If it's missing, reload the window (`Cmd+Shift+P` → `Reload Window`).

All cells talk to `http://localhost:8000` — the same mini-SIEM container seeded with a realistic multi-stage attack plus normal background traffic.


In [ ]:
import httpx
from collections import Counter, defaultdict
from datetime import datetime

SIEM = 'http://localhost:8000'

# Sanity check: can we reach the SIEM?
health = httpx.get(f'{SIEM}/health').json()
print('SIEM health:', health)

dashboard = httpx.get(f'{SIEM}/dashboard').json()
print('Dashboard:', dashboard)


## The classification taxonomy

When you close an incident you **must** classify it. This trains the ML and gives engineering a signal to tune rules.

| Classification (Sentinel API value) | The activity really happened? | Was it malicious? | Example |
|---|---|---|---|
| **True Positive** — *suspicious activity* (`TruePositive`) | ✅ Yes | ✅ Yes | Confirmed phishing → credential theft |
| **Benign Positive** — *suspicious but expected* (`BenignPositive`) | ✅ Yes | ❌ No | Authorised pen-test; an admin genuinely did run PsExec |
| **False Positive** (`FalsePositive`) | ❌ No | — | The detection logic was wrong, or the data was inaccurate |
| **Undetermined** (`Undetermined`) | ❓ | ❓ | Not enough evidence; you closed it after exhausting leads |

The two-column test is the whole trick: **"did it happen?"** then **"was it bad?"**

- Happened **and** bad → **True Positive**.
- Happened but **not** bad → **Benign Positive**. *(The activity was real. Do not call this
  a false positive — the detection worked correctly.)*
- Did **not** happen → **False Positive**. In the Sentinel portal this splits into two
  reasons: *incorrect alert logic* and *inaccurate data*. Pick the one that tells the
  detection engineer what to fix.
- Genuinely unknowable → **Undetermined**. Legitimate, but if half your queue is
  Undetermined your data coverage is the problem.

> **Classification is not derived from severity.** A Medium "phishing email delivered" that
> really was a phish is a **True Positive**; a Critical alert on your own red team is a
> **Benign Positive**. Deciding by severity is exactly the anti-pattern this notebook exists
> to kill — and the cell below deliberately classifies from *evidence*, not from severity.

Closing without classifying is the single most common SOC anti-pattern; classifying from
severity instead of evidence is the second.


## ❌ Bad: silent close

The bad analyst just flips the status to `Closed`. No comment. No classification. No one learns anything.


In [ ]:
incidents = httpx.get(f'{SIEM}/incidents').json()

# We'll demonstrate on a lower-severity incident so we don't close the real attack
target = next((i for i in incidents if i['severity'] != 'High' and i['status'] == 'New'), incidents[-1])
print(f"Demo target: {target['id']}  {target['title']}")

# BAD: just close it
httpx.patch(f"{SIEM}/incidents/{target['id']}", json={'status': 'Closed'})

closed = httpx.get(f"{SIEM}/incidents/{target['id']}").json()
print(f"\n❌ status={closed['status']}  classification={closed['classification']!r}  comments={len(__import__('json').loads(closed['comments']))} entries")


## ✅ Best: assign → comment → classify → close

A mature close does four things, in this order:

1. **Assign** the incident to a named analyst (accountability).
2. **Comment** with the timeline and the "why" of the classification (institutional memory).
3. **Classify** so the detection engine can learn.
4. **Close** the incident.

Our mini-SIEM exposes all four through `PATCH /incidents/{id}`. That endpoint is **not idempotent** — every call appends to comments and overwrites the other fields — so below we read the incident first and only update the fields that need updating.


In [ ]:
def mature_close(incident_id: str, analyst: str, summary: str, classification: str):
    current = httpx.get(f'{SIEM}/incidents/{incident_id}').json()

    # 1. Assign (only if not yet assigned — keeps reruns clean)
    if not current.get('assigned_to'):
        httpx.patch(f'{SIEM}/incidents/{incident_id}', json={'assigned_to': analyst})

    # 2. Add the summary comment (only once, to avoid duplicates on rerun)
    import json as _json
    existing = _json.loads(current.get('comments') or '[]')
    if not any(summary == c.get('text') for c in existing):
        httpx.patch(f'{SIEM}/incidents/{incident_id}', json={'comment': summary})

    # 3. Classify (only if not already classified)
    if not current.get('classification'):
        httpx.patch(f'{SIEM}/incidents/{incident_id}', json={'classification': classification})

    # 4. Close
    httpx.patch(f'{SIEM}/incidents/{incident_id}', json={'status': 'Closed'})

    return httpx.get(f'{SIEM}/incidents/{incident_id}').json()

import json as _json


def classify_from_evidence(incident):
    """Decide the classification from what the EVIDENCE shows, never from severity.

    We apply the two-column test:
      1. Did the activity actually happen?  (is there evidence in the logs?)
      2. Was it malicious, or expected?     (is the actor/tool on an approved list?)
    """
    detail = httpx.get(f"{SIEM}/incidents/{incident['id']}").json()
    alerts = detail.get('alerts', [])

    # Anything the SOC has pre-approved: red team, scanners, sanctioned admin tooling.
    APPROVED_ACTORS = {'redteam@contoso.com', 'svc-vulnscan@contoso.com'}
    APPROVED_SOURCE_IPS = {'10.99.0.0/16'}  # pen-test range, illustrative

    evidence_rows, actors = 0, set()
    for a in alerts:
        rows = _json.loads(a['evidence'] or '[]')
        evidence_rows += len(rows)
        for ev in rows:
            for k in ('UserPrincipalName', 'AccountName', 'InitiatingUser'):
                if ev.get(k):
                    actors.add(ev[k])

    # 1. No supporting evidence at all -> the detection fired on nothing real.
    if evidence_rows == 0:
        return 'FalsePositive', 'No supporting events found; the rule matched on empty/incorrect data.'

    # 2. Real activity, but performed by a pre-approved actor -> expected.
    if actors & APPROVED_ACTORS:
        return 'BenignPositive', f'Activity is real but performed by approved actor(s): {sorted(actors & APPROVED_ACTORS)}.'

    # 3. Real activity by a non-approved actor, corroborated by several events.
    if evidence_rows >= 2:
        return 'TruePositive', f'{evidence_rows} corroborating events from non-approved actor(s) {sorted(actors) or "n/a"}.'

    # 4. A single event, nothing to corroborate it, nothing to exonerate it.
    return 'Undetermined', 'Single uncorroborated event; no further telemetry available to confirm or refute.'


# Run the classifier over EVERY incident so you always see its reasoning, then only
# mutate the ones still open. (This notebook is safe to re-run; on a second pass the
# incidents are already Closed, but the verdicts below are still computed and shown.)
print(f"{'incident':<12} {'sev':<8} {'verdict':<15} status")
print('-' * 100)
for inc in httpx.get(f'{SIEM}/incidents').json():
    cls, why = classify_from_evidence(inc)

    if inc['status'] == 'New':
        result = mature_close(
            inc['id'],
            analyst='analyst1@contoso.com',
            summary=f"Triaged {inc['title']}. {why} Classified as {cls}.",
            classification=cls,
        )
        state = f"closed now, assigned to {result['assigned_to']}"
    else:
        state = (f"already {inc['status']} (stored classification="
                 f"{inc['classification']!r}) — not modified")

    print(f"{inc['id']:<12} {inc['severity']:<8} {cls:<15} {state}")
    print(f"{'':<12} evidence: {why}")
    print(f"{'':<12} title:    {inc['title']}")

# --- Audit: where does the STORED classification disagree with the evidence? ---
print()
print('=== Classification audit: stored value vs evidence-based verdict ===')
disagreements = 0
for inc in httpx.get(f'{SIEM}/incidents').json():
    verdict, why = classify_from_evidence(inc)
    stored = inc.get('classification')
    if stored is None:
        disagreements += 1
        print(f"  ⚠️  {inc['id']} sev={inc['severity']:<7} stored=<UNCLASSIFIED>  evidence says {verdict}")
        print(f"      → someone closed this without classifying. Detection engineering learns nothing.")
    elif stored != verdict:
        disagreements += 1
        print(f"  ❌ {inc['id']} sev={inc['severity']:<7} stored={stored:<15} evidence says {verdict}")
        print(f"      → {why}")
        print(f"      → this is what classifying by SEVERITY instead of EVIDENCE produces.")
if not disagreements:
    print('  ✅ Every incident carries a classification consistent with its evidence.')

print()
print('👀 Severity and classification are independent axes and must not be derived from')
print('   each other:')
print('     Severity       = "how bad would this be IF it is real"   (set by the rule author)')
print('     Classification = "was it real, and was it malicious"     (set by the analyst,')
print('                       from evidence, AFTER the investigation)')
print('   A Medium "phishing email delivered" that really was a phish is a TRUE POSITIVE.')
print('   A Critical alert on your own authorised red team is a BENIGN POSITIVE.')
print('   Any rule of the form "High => TruePositive" is a bug in your process, not a shortcut.')


## Tuning: the forgotten step

If you classify an incident as **False Positive**, the job isn't done — you should also **tune the rule** so it stops firing on the same false pattern. Otherwise you'll keep closing the same FP over and over.

Common tuning moves in Sentinel/Defender XDR:

| Tuning move | When to use |
|---|---|
| **Raise the threshold** | Rule fires on benign bursts of activity |
| **Add an exclusion filter** | A specific admin tool or scanner is triggering it |
| **Suppress for an entity** | A specific service account generates expected traffic |
| **Lower the severity** | The pattern is suspicious but rarely a real threat |
| **Disable the rule** | The rule is obsolete (only as a last resort) |

Let's simulate tuning: show that a `False Positive` incident points us at a rule we should adjust.


In [ ]:
fps = [i for i in httpx.get(f'{SIEM}/incidents').json() if i.get('classification') == 'FalsePositive']
print(f'False-positive incidents: {len(fps)}')
if fps:
    for i in fps:
        print(f"  → tune rule behind: {i['title']}")
else:
    print('None in this seed data — but in a real SOC, every FP should trigger a rule review within the week.')

# Show the rules that would be candidates for tuning review
rules = httpx.get(f'{SIEM}/rules').json()
print('\nActive rules (candidates for tuning if misfiring):')
for r in rules:
    print(f"  {r['name']:<34}  sev={r['severity']:<6} threshold={r['threshold']:<3} window={r['window_minutes']}m")


## What you just did (SC-200 mapping)

| You did... | Real portal equivalent |
|---|---|
| `PATCH /incidents/{id}` with `status/assigned_to/classification/comment` | Incident page → Manage incident panel |
| Guarded against duplicate comments on rerun | Real portals don't dedupe for you either — discipline matters |
| Listed rule parameters for tuning | Analytics rule → Edit → Query logic / thresholds |

### Exam tips

- Know the **four classifications** by name and when to use each. Apply the two-column
  test: *did it happen?* then *was it malicious?*
- **Benign Positive ≠ False Positive.** Benign Positive = the activity was real but
  expected (the detection worked). False Positive = the detection was wrong. Getting
  this backwards is the most commonly missed SC-200 question in this domain.
- Classification comes from **evidence**, never from severity.
- "Close" is **not** a classification — it's a status. Always classify.
- **False Positive ⇒ tune the rule.** Closing FPs without tuning is how alert fatigue starts.
  **Benign Positive ⇒ add an exclusion or suppression**, not a threshold change.
- In Defender XDR, closing an incident closes all underlying alerts automatically (and
  resolving all of an incident's alerts closes the incident).

➡️ Next: [04 — Automated response & IOCs](04_automated_response.ipynb)


---
## ✅ Self-check

1. Your red team ran Mimikatz on a lab host as part of an authorised exercise, and Defender
   raised a Critical incident. How do you classify it, and what tuning action follows?
2. A rule fires on "impossible travel" for a user whose VPN egress moved between regions.
   Classification? Tuning action?
3. What is the difference between a Benign Positive and a False Positive, in one sentence?
4. When is `Undetermined` the honest answer, and what does a queue full of them tell you?
5. You close an incident in Defender XDR. What happens to its alerts?
6. Which is a status and which is a classification: `Active`, `TruePositive`, `Closed`,
   `BenignPositive`?

In [ ]:
answers = """
1. BENIGN POSITIVE (suspicious but expected). The activity genuinely happened and the
   detection was correct -- it simply was not hostile. Tuning action: add an EXCLUSION
   or SUPPRESSION for the red-team accounts / lab host range (or have the red team
   pre-register the exercise window), NOT a threshold change and definitely not
   disabling the Mimikatz rule.

2. BENIGN POSITIVE again -- the sign-ins really did come from two distant IPs; the
   inference "impossible travel" is what is wrong for this environment. Tuning:
   add the VPN egress ranges as named/trusted locations so the rule stops treating
   them as travel. (If the rule had fired on sign-ins that never happened -- bad data,
   duplicated logs -- that would be a False Positive instead.)

3. Benign Positive = it happened but it was expected; False Positive = it did not
   happen as described, the detection or the data was wrong.

4. When you have genuinely exhausted the available telemetry and can neither confirm
   nor refute the activity -- e.g. a single sign-in anomaly on a device with no EDR.
   A queue full of Undetermined is a DATA COVERAGE problem, not an analyst problem:
   you are missing a connector, a log source, or retention.

5. All alerts attached to the incident are closed/resolved with it. The reverse also
   holds: resolving every alert in an incident closes the incident.

6. Statuses: Active, Closed (also New).
   Classifications: TruePositive, BenignPositive (also FalsePositive, Undetermined).
   Every closed incident must carry both a status AND a classification.
"""
print(answers)